# Statistical Validation Framework for GI Hybrid Model
## 5-Run Repeated Evaluation with Full Statistical Analysis

**Paper**: Hybrid Deep Convolution Learning Model Integrated with XAI for GI Tract Classification

**Methodology**: Adapted from Alabdullah & Sankaranarayanan (2026), Frontiers in AI

**Tests implemented**:
1. Mean ± SD over 5 repeated runs
2. Paired t-test
3. McNemar's test
4. Wilcoxon Signed-Rank test
5. Cohen's d Effect Size
6. Bootstrap Confidence Intervals (n=10,000)

**Estimated Runtime**: ~2.5 hours on T4 GPU


## Cell 1: Install Required Packages


In [ ]:
!pip install -q catboost optuna einops timm scipy statsmodels


## Cell 2: Import All Libraries


In [ ]:
import os
import sys
import time
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import tensorflow as tf

from catboost import CatBoostClassifier
import optuna
from optuna.samplers import TPESampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support,
    log_loss, confusion_matrix, f1_score
)

from scipy import stats
from scipy.stats import wilcoxon

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## Cell 3: Mount Drive & Prepare Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Download Kvasir-v2 dataset (skip if already downloaded)
import subprocess
result = subprocess.run(['ls', 'kvasir-dataset.zip'], capture_output=True)
if result.returncode != 0:
    print("Downloading dataset...")
    !pip install -q kaggle
    !mkdir -p ~/.kaggle
    # UPDATE THIS PATH to where your kaggle.json is on Drive
    !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/ 2>/dev/null || echo "kaggle.json not found, trying direct download"
    !chmod 600 ~/.kaggle/kaggle.json 2>/dev/null
    !kaggle datasets download -d meetnagadia/kvasir-dataset 2>/dev/null || echo "Download via kaggle failed"
else:
    print("Dataset zip already exists, skipping download")

# Unzip (overwrite without asking)
!unzip -qo kvasir-dataset.zip -d kvasir-data
print("✅ Dataset ready")
!ls kvasir-data/


## Cell 4: Copy Model Files from Drive

⚠️ **UPDATE THE PATHS BELOW** to match your Google Drive folder structure.

Run `!ls /content/drive/MyDrive/` first to find your files.


In [ ]:
# First, check what's on your Drive
!ls /content/drive/MyDrive/

# UPDATE THESE PATHS to match your Drive structure:
# Example: !cp /content/drive/MyDrive/YourFolder/hybrid_model_best.pth /content/

# Hybrid model weights
!cp /content/drive/MyDrive/hybrid_model_best.pth /content/ 2>/dev/null || echo "⚠️ UPDATE PATH for hybrid_model_best.pth"

# MobileViT model code
!cp -r /content/drive/MyDrive/mobvit /content/ 2>/dev/null || echo "⚠️ UPDATE PATH for mobvit/"

# EfficientNet weights
!cp -r /content/drive/MyDrive/effnet /content/ 2>/dev/null || echo "⚠️ UPDATE PATH for effnet/"

# Verify files exist
print("\n📁 Checking files:")
for f in ['hybrid_model_best.pth', 'mobvit/model.py',
          'effnet/model_checkpoints/efficientnet_v2m_best_model.weights.h5']:
    status = "✅" if os.path.exists(f) else "❌ MISSING"
    print(f"  {status} {f}")


## Cell 5: Configuration Constants


In [ ]:
RANDOM_SEEDS = [42, 123, 456, 789, 1024]  # 5 runs
N_RUNS = len(RANDOM_SEEDS)
NUM_CLASSES = 8
IMAGE_SIZE = 224
BATCH_SIZE = 32
OPTUNA_TRIALS = 50   # Full optimization per run
BOOTSTRAP_N = 10000  # Bootstrap resamples
ALPHA = 0.05         # Significance level

CLASS_NAMES = [
    'dyed-lifted-polyps', 'dyed-resection-margins', 'esophagitis',
    'normal-cecum', 'normal-pylorus', 'normal-z-line', 'polyps',
    'ulcerative-colitis'
]

print(f"Configuration:")
print(f"  Random Seeds:        {RANDOM_SEEDS}")
print(f"  Optuna trials/run:   {OPTUNA_TRIALS}")
print(f"  Bootstrap resamples: {BOOTSTRAP_N}")
print(f"  Significance level:  α = {ALPHA}")


## Cell 6: Model Architecture Definitions


In [ ]:
sys.path.append('./mobvit')
from model import MobileViT

def create_mobilevit(num_classes=8):
    return MobileViT(
        image_size=224, num_classes=num_classes,
        dims=[96, 192, 384],
        channels=[16, 32, 64, 128, 160, 192, 256, 320, 384, 512],
        depths=[2, 4, 3]
    )

def create_efficientnet(num_classes=8):
    base_model = tf.keras.applications.EfficientNetV2M(
        input_shape=(224, 224, 3), include_top=False, weights="imagenet"
    )
    base_model.trainable = True
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = tf.keras.applications.efficientnet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

class HybridGIModel(nn.Module):
    def __init__(self, num_classes=8):
        super(HybridGIModel, self).__init__()
        self.mobilevit = create_mobilevit(num_classes=num_classes)
        self.fusion_weights = nn.Parameter(torch.tensor([0.5, 0.5]))
        self.fusion_fc = nn.Sequential(
            nn.Linear(num_classes * 2, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, num_classes)
        )
        self.num_classes = num_classes
        self.use_advanced_fusion = True

    def forward(self, x, efficientnet_logits=None, return_features=False):
        mobilevit_logits = self.mobilevit(x)
        if efficientnet_logits is None:
            return mobilevit_logits
        weights = F.softmax(self.fusion_weights, dim=0)
        combined = torch.cat([mobilevit_logits, efficientnet_logits], dim=1)
        if return_features:
            return combined
        if self.use_advanced_fusion:
            output = self.fusion_fc(combined)
        else:
            output = weights[0] * mobilevit_logits + weights[1] * efficientnet_logits
        return output

print("✅ Model classes defined")


## Cell 7: Dataset Class & Data Loading Utilities


In [ ]:
class KvasirDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def load_dataset_paths(data_dir):
    image_paths, labels = [], []
    class_names = sorted(os.listdir(data_dir))
    class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
    for class_name in class_names:
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(os.path.join(class_dir, img_name))
                labels.append(class_to_idx[class_name])
    return image_paths, labels, class_names

print("✅ Dataset class defined")


## Cell 8: Load Frozen Model Weights


In [ ]:
print("📂 Loading frozen model weights...")

hybrid_model = HybridGIModel(num_classes=NUM_CLASSES).to(device)
hybrid_model.load_state_dict(torch.load('hybrid_model_best.pth', map_location=device))
hybrid_model.eval()
print("  ✅ Hybrid model loaded")

efficientnet_model = create_efficientnet(num_classes=NUM_CLASSES)
efficientnet_model.load_weights(
    "effnet/model_checkpoints/efficientnet_v2m_best_model.weights.h5"
)
print("  ✅ EfficientNet loaded")

print("\n✅ All frozen model weights loaded successfully")


## Cell 9: Feature Extraction & Evaluation Functions


In [ ]:
def extract_hybrid_features(model, efficientnet_model, data_loader, device):
    """Extract 16-dim hybrid features from frozen backbones."""
    all_features = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)

            images_np = images.cpu().numpy().transpose(0, 2, 3, 1)
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            images_np = images_np * std + mean
            images_np = np.clip(images_np * 255, 0, 255)

            eff_logits = efficientnet_model.predict(images_np, verbose=0)
            eff_logits = torch.from_numpy(eff_logits).float().to(device)

            hybrid_features = model(images, eff_logits, return_features=True)
            all_features.append(hybrid_features.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.vstack(all_features), np.array(all_labels)


def evaluate_softmax(model, efficientnet_model, data_loader, device):
    """Evaluate the UNOPTIMIZED hybrid model (softmax classifier)."""
    all_preds = []
    all_probs = []
    all_labels = []

    start_time = time.time()
    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)

            images_np = images.cpu().numpy().transpose(0, 2, 3, 1)
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            images_np = images_np * std + mean
            images_np = np.clip(images_np * 255, 0, 255)

            eff_logits = efficientnet_model.predict(images_np, verbose=0)
            eff_logits = torch.from_numpy(eff_logits).float().to(device)

            output = model(images, eff_logits, return_features=False)
            probs = F.softmax(output, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)

            all_probs.append(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    elapsed = time.time() - start_time
    return np.array(all_preds), np.vstack(all_probs), np.array(all_labels), elapsed

print("✅ Helper functions defined")


## Cell 10: Optuna TPE + CatBoost Optimization Function


In [ ]:
def optimize_and_evaluate_catboost(train_features, train_labels,
                                    val_features, val_labels,
                                    test_features, test_labels,
                                    seed):
    """Run full Optuna TPE optimization + CatBoost training for one seed."""
    start_time = time.time()

    def objective(trial):
        params = {
            'iterations': trial.suggest_int('iterations', 50, 500),
            'depth': trial.suggest_int('depth', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.01, 10.0, log=True),
            'border_count': trial.suggest_int('border_count', 32, 255),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
            'random_seed': seed,
            'verbose': 0,
            'task_type': 'GPU' if torch.cuda.is_available() else 'CPU',
            'loss_function': 'MultiClass',
            'eval_metric': 'MultiClass',
        }

        model = CatBoostClassifier(**params)
        model.fit(
            train_features, train_labels,
            eval_set=(val_features, val_labels),
            verbose=0, early_stopping_rounds=50
        )

        val_preds_proba = model.predict_proba(val_features)
        val_loss = log_loss(val_labels, val_preds_proba)
        return val_loss

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    sampler = TPESampler(seed=seed)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

    best_params = study.best_params
    best_params['random_seed'] = seed
    best_params['verbose'] = 0
    best_params['task_type'] = 'GPU' if torch.cuda.is_available() else 'CPU'
    best_params['loss_function'] = 'MultiClass'
    best_params['eval_metric'] = 'MultiClass'

    final_model = CatBoostClassifier(**best_params)
    final_model.fit(
        train_features, train_labels,
        eval_set=(val_features, val_labels),
        verbose=0, early_stopping_rounds=50
    )

    test_preds = final_model.predict(test_features).flatten().astype(int)
    test_probs = final_model.predict_proba(test_features)

    elapsed = time.time() - start_time

    accuracy = accuracy_score(test_labels, test_preds)
    auroc = roc_auc_score(test_labels, test_probs, multi_class='ovr', average='macro')
    logloss = log_loss(test_labels, test_probs)
    f1 = f1_score(test_labels, test_preds, average='weighted')
    precision, recall, _, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='weighted', zero_division=0
    )

    return {
        'predictions': test_preds,
        'probabilities': test_probs,
        'accuracy': accuracy,
        'auroc': auroc,
        'log_loss': logloss,
        'f1_score': f1,
        'precision': precision,
        'recall': recall,
        'time': elapsed,
        'best_params': best_params,
        'best_val_loss': study.best_value
    }

print("✅ Optuna + CatBoost function defined")


## Cell 11: Load Dataset


In [ ]:
print("📊 Loading Kvasir-v2 dataset...")

dataset_path = None
for p in ['kvasir-data/kvasir-dataset-v2/kvasir-dataset-v2',
          'kvasir-data/kvasir-dataset-v2',
          'kvasir-data/kvasir-dataset',
          'kvasir-dataset-v2',
          'mobvit/kvasir-dataset-v2/kvasir-dataset-v2',
          'mobvit/kvasir-dataset-v2']:
    if os.path.exists(p):
        # Check it actually has class folders
        subdirs = [d for d in os.listdir(p) if os.path.isdir(os.path.join(p, d))]
        if len(subdirs) >= 8:
            dataset_path = p
            break

if dataset_path is None:
    print("❌ Dataset not found! Listing kvasir-data/ to debug:")
    !find kvasir-data/ -maxdepth 3 -type d | head -20
else:
    all_image_paths, all_labels, class_names = load_dataset_paths(dataset_path)
    print(f"✅ Dataset loaded from: {dataset_path}")
    print(f"   Total images: {len(all_image_paths)}")
    print(f"   Classes ({len(class_names)}): {class_names}")


## Cell 12: ⭐ MAIN 5-RUN EVALUATION LOOP ⭐

**⏱️ This cell takes ~2.5 hours on T4 GPU.**

Intermediate results are saved after each run to prevent data loss.

Each run:
1. Splits data with a different random seed
2. Evaluates Softmax (unoptimized) classifier
3. Extracts hybrid features
4. Runs 50-trial Optuna TPE optimization + CatBoost


In [ ]:
print("=" * 80)
print("STARTING 5-RUN STATISTICAL EVALUATION".center(80))
print("=" * 80)

softmax_results = {
    'accuracy': [], 'auroc': [], 'log_loss': [], 'f1_score': [],
    'precision': [], 'recall': [], 'time': [],
    'per_run_preds': [], 'per_run_labels': []
}
catboost_results = {
    'accuracy': [], 'auroc': [], 'log_loss': [], 'f1_score': [],
    'precision': [], 'recall': [], 'time': [],
    'per_run_preds': [], 'per_run_labels': [],
    'best_params': [], 'best_val_loss': []
}

for run_idx, seed in enumerate(RANDOM_SEEDS):
    print(f"\n{'='*80}")
    print(f"  RUN {run_idx + 1}/{N_RUNS} — Seed: {seed}")
    print(f"{'='*80}")
    run_start = time.time()

    # Step 1: Split data with this seed
    print(f"  [1/4] Splitting data (seed={seed})...")
    train_paths, temp_paths, train_labels_split, temp_labels = train_test_split(
        all_image_paths, all_labels, test_size=0.3,
        random_state=seed, stratify=all_labels
    )
    val_paths, test_paths, val_labels_split, test_labels_split = train_test_split(
        temp_paths, temp_labels, test_size=0.5,
        random_state=seed, stratify=temp_labels
    )
    print(f"    Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

    train_dataset = KvasirDataset(train_paths, train_labels_split, transform=transform)
    val_dataset = KvasirDataset(val_paths, val_labels_split, transform=transform)
    test_dataset = KvasirDataset(test_paths, test_labels_split, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # Step 2: Evaluate Softmax (Unoptimized)
    print(f"  [2/4] Evaluating SOFTMAX classifier...")
    sm_preds, sm_probs, sm_labels, sm_time = evaluate_softmax(
        hybrid_model, efficientnet_model, test_loader, device
    )

    sm_acc = accuracy_score(sm_labels, sm_preds)
    sm_auroc = roc_auc_score(sm_labels, sm_probs, multi_class='ovr', average='macro')
    sm_ll = log_loss(sm_labels, sm_probs)
    sm_f1 = f1_score(sm_labels, sm_preds, average='weighted')
    sm_prec, sm_rec, _, _ = precision_recall_fscore_support(
        sm_labels, sm_preds, average='weighted', zero_division=0
    )

    softmax_results['accuracy'].append(sm_acc)
    softmax_results['auroc'].append(sm_auroc)
    softmax_results['log_loss'].append(sm_ll)
    softmax_results['f1_score'].append(sm_f1)
    softmax_results['precision'].append(sm_prec)
    softmax_results['recall'].append(sm_rec)
    softmax_results['time'].append(sm_time)
    softmax_results['per_run_preds'].append(sm_preds)
    softmax_results['per_run_labels'].append(sm_labels)

    print(f"    Softmax -> Acc: {sm_acc*100:.2f}%, AUROC: {sm_auroc:.4f}, "
          f"F1: {sm_f1:.4f}, LogLoss: {sm_ll:.4f}")

    # Step 3: Extract features for CatBoost
    print(f"  [3/4] Extracting hybrid features...")
    train_features, train_feat_labels = extract_hybrid_features(
        hybrid_model, efficientnet_model, train_loader, device
    )
    val_features, val_feat_labels = extract_hybrid_features(
        hybrid_model, efficientnet_model, val_loader, device
    )
    test_features, test_feat_labels = extract_hybrid_features(
        hybrid_model, efficientnet_model, test_loader, device
    )

    # Step 4: Optuna + CatBoost
    print(f"  [4/4] Running Optuna TPE ({OPTUNA_TRIALS} trials, seed={seed})...")
    cb_result = optimize_and_evaluate_catboost(
        train_features, train_feat_labels,
        val_features, val_feat_labels,
        test_features, test_feat_labels,
        seed=seed
    )

    catboost_results['accuracy'].append(cb_result['accuracy'])
    catboost_results['auroc'].append(cb_result['auroc'])
    catboost_results['log_loss'].append(cb_result['log_loss'])
    catboost_results['f1_score'].append(cb_result['f1_score'])
    catboost_results['precision'].append(cb_result['precision'])
    catboost_results['recall'].append(cb_result['recall'])
    catboost_results['time'].append(cb_result['time'])
    catboost_results['per_run_preds'].append(cb_result['predictions'])
    catboost_results['per_run_labels'].append(test_feat_labels)
    catboost_results['best_params'].append(cb_result['best_params'])
    catboost_results['best_val_loss'].append(cb_result['best_val_loss'])

    run_elapsed = time.time() - run_start
    print(f"    CatBoost -> Acc: {cb_result['accuracy']*100:.2f}%, "
          f"AUROC: {cb_result['auroc']:.4f}, F1: {cb_result['f1_score']:.4f}, "
          f"LogLoss: {cb_result['log_loss']:.4f}")
    print(f"    Best Val Loss: {cb_result['best_val_loss']:.4f}")
    print(f"  ✅ Run {run_idx + 1} done in {run_elapsed/60:.1f} min")

    # Save intermediate results to prevent data loss
    intermediate = {
        'completed_runs': run_idx + 1,
        'softmax_accuracy': softmax_results['accuracy'],
        'softmax_auroc': softmax_results['auroc'],
        'softmax_log_loss': softmax_results['log_loss'],
        'softmax_f1': softmax_results['f1_score'],
        'softmax_precision': softmax_results['precision'],
        'softmax_recall': softmax_results['recall'],
        'softmax_time': softmax_results['time'],
        'catboost_accuracy': catboost_results['accuracy'],
        'catboost_auroc': catboost_results['auroc'],
        'catboost_log_loss': catboost_results['log_loss'],
        'catboost_f1': catboost_results['f1_score'],
        'catboost_precision': catboost_results['precision'],
        'catboost_recall': catboost_results['recall'],
        'catboost_time': catboost_results['time'],
        'catboost_best_val_loss': catboost_results['best_val_loss'],
    }
    with open('statistical_intermediate_results.json', 'w') as f:
        json.dump(intermediate, f, indent=2)
    # Also save to Drive
    with open('/content/drive/MyDrive/statistical_intermediate_results.json', 'w') as f:
        json.dump(intermediate, f, indent=2)
    print(f"  💾 Intermediate results saved (Run {run_idx + 1}/{N_RUNS})")

print("\n" + "=" * 80)
print("✅ ALL 5 RUNS COMPLETED SUCCESSFULLY!".center(80))
print("=" * 80)


## Cell 13: TABLE A — Mean ± SD Performance Variability
*Matching the format of Tables 15 and 19 from Alabdullah & Sankaranarayanan (2026)*


In [ ]:
print("=" * 80)
print("TABLE A: PERFORMANCE VARIABILITY (MEAN +/- SD OVER 5 RUNS)".center(80))
print("=" * 80)

metrics_list = ['accuracy', 'auroc', 'log_loss', 'f1_score', 'precision', 'recall', 'time']
metric_labels = {
    'accuracy': 'Accuracy (%)',
    'auroc': 'AUROC',
    'log_loss': 'Log Loss',
    'f1_score': 'F1-Score',
    'precision': 'Precision',
    'recall': 'Recall',
    'time': 'Time (s)'
}

table_data = []
for metric in metrics_list:
    sm_vals = np.array(softmax_results[metric])
    cb_vals = np.array(catboost_results[metric])

    if metric in ['accuracy', 'precision', 'recall', 'f1_score']:
        sm_str = f"{sm_vals.mean()*100:.2f} +/- {sm_vals.std()*100:.2f}%"
        cb_str = f"{cb_vals.mean()*100:.2f} +/- {cb_vals.std()*100:.2f}%"
        diff = f"+{(cb_vals.mean() - sm_vals.mean())*100:.2f}%"
    elif metric == 'auroc':
        sm_str = f"{sm_vals.mean():.4f} +/- {sm_vals.std():.4f}"
        cb_str = f"{cb_vals.mean():.4f} +/- {cb_vals.std():.4f}"
        diff = f"+{cb_vals.mean() - sm_vals.mean():.4f}"
    elif metric == 'log_loss':
        sm_str = f"{sm_vals.mean():.4f} +/- {sm_vals.std():.4f}"
        cb_str = f"{cb_vals.mean():.4f} +/- {cb_vals.std():.4f}"
        diff = f"{cb_vals.mean() - sm_vals.mean():.4f}"
    else:
        sm_str = f"{sm_vals.mean():.2f} +/- {sm_vals.std():.2f}"
        cb_str = f"{cb_vals.mean():.2f} +/- {cb_vals.std():.2f}"
        diff = "---"

    table_data.append({
        'Metric': metric_labels[metric],
        'Softmax (Mean +/- SD)': sm_str,
        'CatBoost (Mean +/- SD)': cb_str,
        'Improvement': diff
    })

df_table_a = pd.DataFrame(table_data)
print(df_table_a.to_string(index=False))

# Per-run breakdown
print("\n" + "-" * 80)
print("Per-Run Breakdown:")
print(f"{'Run':<6} {'Seed':<8} {'SM Acc':<12} {'CB Acc':<12} {'SM AUROC':<12} {'CB AUROC':<12}")
print("-" * 62)
for i, seed in enumerate(RANDOM_SEEDS):
    print(f"{i+1:<6} {seed:<8} "
          f"{softmax_results['accuracy'][i]*100:.2f}%{'':>4} "
          f"{catboost_results['accuracy'][i]*100:.2f}%{'':>4} "
          f"{softmax_results['auroc'][i]:.4f}{'':>4} "
          f"{catboost_results['auroc'][i]:.4f}")


## Cell 14: TABLE B — Paired t-test
*Matching the methodology of the reference paper (paired statistical T-tests)*


In [ ]:
print("=" * 80)
print("TABLE B: PAIRED t-TEST".center(80))
print("=" * 80)

ttest_results = []
for metric in ['accuracy', 'auroc', 'log_loss', 'f1_score', 'precision', 'recall']:
    sm_vals = np.array(softmax_results[metric])
    cb_vals = np.array(catboost_results[metric])

    t_stat, p_value = stats.ttest_rel(cb_vals, sm_vals)

    significant = "Yes" if p_value < ALPHA else "No"
    ttest_results.append({
        'Metric': metric_labels.get(metric, metric),
        't-statistic': f"{t_stat:.4f}",
        'p-value': f"{p_value:.6f}",
        f'Significant (a={ALPHA})': significant
    })

df_ttest = pd.DataFrame(ttest_results)
print(df_ttest.to_string(index=False))

print("\nInterpretation:")
for row in ttest_results:
    p = float(row['p-value'])
    if p < 0.001:
        sig = "highly significant (p < 0.001)"
    elif p < 0.01:
        sig = "significant (p < 0.01)"
    elif p < 0.05:
        sig = "significant (p < 0.05)"
    else:
        sig = f"not significant (p = {p:.4f})"
    print(f"  {row['Metric']}: CatBoost vs Softmax is {sig}")


## Cell 15: TABLE C — Cohen's d Effect Size


In [ ]:
print("=" * 80)
print("TABLE C: COHEN'S d EFFECT SIZE".center(80))
print("=" * 80)

def cohens_d(x, y):
    diff = x - y
    return diff.mean() / diff.std(ddof=1)

def interpret_d(d):
    d_abs = abs(d)
    if d_abs < 0.2: return "Negligible"
    elif d_abs < 0.5: return "Small"
    elif d_abs < 0.8: return "Medium"
    else: return "Large"

effect_results = []
for metric in ['accuracy', 'auroc', 'f1_score']:
    sm_vals = np.array(softmax_results[metric])
    cb_vals = np.array(catboost_results[metric])
    d = cohens_d(cb_vals, sm_vals)
    effect_results.append({
        'Metric': metric_labels.get(metric, metric),
        "Cohen's d": f"{d:.4f}",
        'Interpretation': interpret_d(d)
    })

df_effect = pd.DataFrame(effect_results)
print(df_effect.to_string(index=False))


## Cell 16: TABLE D — Wilcoxon Signed-Rank Test (Non-parametric)


In [ ]:
print("=" * 80)
print("TABLE D: WILCOXON SIGNED-RANK TEST".center(80))
print("=" * 80)

wilcoxon_results = []
for metric in ['accuracy', 'auroc', 'f1_score']:
    sm_vals = np.array(softmax_results[metric])
    cb_vals = np.array(catboost_results[metric])

    try:
        w_stat, w_p = wilcoxon(cb_vals, sm_vals, alternative='greater')
    except ValueError:
        w_stat, w_p = float('nan'), float('nan')

    significant = "Yes" if w_p < ALPHA else "No"
    wilcoxon_results.append({
        'Metric': metric_labels.get(metric, metric),
        'W-statistic': f"{w_stat:.4f}" if not np.isnan(w_stat) else "N/A",
        'p-value': f"{w_p:.6f}" if not np.isnan(w_p) else "N/A",
        f'Significant (a={ALPHA})': significant
    })

df_wilcoxon = pd.DataFrame(wilcoxon_results)
print(df_wilcoxon.to_string(index=False))


## Cell 17: TABLE E — McNemar's Test (Error Pattern Analysis)


In [ ]:
print("=" * 80)
print("TABLE E: McNEMAR'S TEST".center(80))
print("=" * 80)

mcnemar_results = []
for run_idx in range(N_RUNS):
    sm_preds = softmax_results['per_run_preds'][run_idx]
    cb_preds = catboost_results['per_run_preds'][run_idx]
    true_labels = softmax_results['per_run_labels'][run_idx]

    sm_correct = (sm_preds == true_labels)
    cb_correct = (cb_preds == true_labels)

    b = int(np.sum(~sm_correct & cb_correct))  # SM wrong, CB right
    c = int(np.sum(sm_correct & ~cb_correct))   # SM right, CB wrong

    if b + c > 0:
        chi2 = (abs(b - c) - 1) ** 2 / (b + c)
        p_value = 1 - stats.chi2.cdf(chi2, df=1)
    else:
        chi2, p_value = 0.0, 1.0

    significant = "Yes" if p_value < ALPHA else "No"
    mcnemar_results.append({
        'Run': f"Run {run_idx+1} (seed={RANDOM_SEEDS[run_idx]})",
        'b (SM wrong, CB right)': b,
        'c (SM right, CB wrong)': c,
        'chi2': f"{chi2:.4f}",
        'p-value': f"{p_value:.6f}",
        'Significant': significant
    })

df_mcnemar = pd.DataFrame(mcnemar_results)
print(df_mcnemar.to_string(index=False))

# Aggregated McNemar's
total_b = sum(r['b (SM wrong, CB right)'] for r in mcnemar_results)
total_c = sum(r['c (SM right, CB wrong)'] for r in mcnemar_results)
if total_b + total_c > 0:
    agg_chi2 = (abs(total_b - total_c) - 1) ** 2 / (total_b + total_c)
    agg_p = 1 - stats.chi2.cdf(agg_chi2, df=1)
else:
    agg_chi2, agg_p = 0.0, 1.0

print(f"\nAggregated McNemar's (all runs):")
print(f"   Total b (SM wrong, CB right): {total_b}")
print(f"   Total c (SM right, CB wrong): {total_c}")
print(f"   chi2 = {agg_chi2:.4f}, p = {agg_p:.6f}")
print(f"   Net: {total_b - total_c} more samples correctly classified by CatBoost")


## Cell 18: TABLE F — Bootstrap Confidence Intervals (n=10,000)


In [ ]:
print("=" * 80)
print(f"TABLE F: BOOTSTRAP CONFIDENCE INTERVALS (n={BOOTSTRAP_N})".center(80))
print("=" * 80)

np.random.seed(42)

bootstrap_results = []
for metric in ['accuracy', 'auroc', 'f1_score']:
    sm_vals = np.array(softmax_results[metric])
    cb_vals = np.array(catboost_results[metric])

    diffs = cb_vals - sm_vals
    boot_diffs = []
    for _ in range(BOOTSTRAP_N):
        boot_sample = np.random.choice(diffs, size=len(diffs), replace=True)
        boot_diffs.append(boot_sample.mean())

    boot_diffs = np.array(boot_diffs)
    ci_lower = np.percentile(boot_diffs, 2.5)
    ci_upper = np.percentile(boot_diffs, 97.5)
    boot_mean = boot_diffs.mean()
    boot_p = np.mean(boot_diffs <= 0)

    if metric in ['accuracy', 'f1_score']:
        ci_str = f"[{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]"
        mean_str = f"+{boot_mean*100:.2f}%"
    else:
        ci_str = f"[{ci_lower:.4f}, {ci_upper:.4f}]"
        mean_str = f"+{boot_mean:.4f}"

    bootstrap_results.append({
        'Metric': metric_labels.get(metric, metric),
        'Mean Delta': mean_str,
        '95% CI': ci_str,
        'Zero excluded': "Yes" if ci_lower > 0 else "No",
        'Bootstrap p': f"{boot_p:.6f}"
    })

df_bootstrap = pd.DataFrame(bootstrap_results)
print(df_bootstrap.to_string(index=False))


## Cell 19: Save All Results as CSVs


In [ ]:
output_dir = 'statistical_validation_results'
os.makedirs(output_dir, exist_ok=True)

df_table_a.to_csv(f'{output_dir}/Table_A_Performance_Variability.csv', index=False)
df_ttest.to_csv(f'{output_dir}/Table_B_Paired_ttest.csv', index=False)
df_effect.to_csv(f'{output_dir}/Table_C_Cohens_d.csv', index=False)
df_wilcoxon.to_csv(f'{output_dir}/Table_D_Wilcoxon.csv', index=False)
df_mcnemar.to_csv(f'{output_dir}/Table_E_McNemars.csv', index=False)
df_bootstrap.to_csv(f'{output_dir}/Table_F_Bootstrap_CI.csv', index=False)

# Per-run raw data
per_run = []
for i, seed in enumerate(RANDOM_SEEDS):
    per_run.append({
        'Run': i+1, 'Seed': seed,
        'SM_Acc': softmax_results['accuracy'][i],
        'SM_AUROC': softmax_results['auroc'][i],
        'SM_LogLoss': softmax_results['log_loss'][i],
        'SM_F1': softmax_results['f1_score'][i],
        'SM_Precision': softmax_results['precision'][i],
        'SM_Recall': softmax_results['recall'][i],
        'SM_Time': softmax_results['time'][i],
        'CB_Acc': catboost_results['accuracy'][i],
        'CB_AUROC': catboost_results['auroc'][i],
        'CB_LogLoss': catboost_results['log_loss'][i],
        'CB_F1': catboost_results['f1_score'][i],
        'CB_Precision': catboost_results['precision'][i],
        'CB_Recall': catboost_results['recall'][i],
        'CB_Time': catboost_results['time'][i],
    })
pd.DataFrame(per_run).to_csv(f'{output_dir}/Per_Run_Raw_Data.csv', index=False)

# Also copy to Drive
!cp -r {output_dir} /content/drive/MyDrive/

print(f"All CSVs saved to {output_dir}/ and copied to Drive:")
for f in sorted(os.listdir(output_dir)):
    print(f"   {f}")


## Cell 20: Generate Publication-Ready Text for Paper


In [ ]:
sm_acc_m = np.mean(softmax_results['accuracy'])*100
sm_acc_s = np.std(softmax_results['accuracy'])*100
cb_acc_m = np.mean(catboost_results['accuracy'])*100
cb_acc_s = np.std(catboost_results['accuracy'])*100
sm_auroc_m = np.mean(softmax_results['auroc'])
sm_auroc_s = np.std(softmax_results['auroc'])
cb_auroc_m = np.mean(catboost_results['auroc'])
cb_auroc_s = np.std(catboost_results['auroc'])
sm_ll_m = np.mean(softmax_results['log_loss'])
sm_ll_s = np.std(softmax_results['log_loss'])
cb_ll_m = np.mean(catboost_results['log_loss'])
cb_ll_s = np.std(catboost_results['log_loss'])
sm_f1_m = np.mean(softmax_results['f1_score'])
sm_f1_s = np.std(softmax_results['f1_score'])
cb_f1_m = np.mean(catboost_results['f1_score'])
cb_f1_s = np.std(catboost_results['f1_score'])

acc_t = float(ttest_results[0]['t-statistic'])
acc_p = float(ttest_results[0]['p-value'])
auroc_t = float(ttest_results[1]['t-statistic'])
auroc_p = float(ttest_results[1]['p-value'])

text = f"""
========================================================================
PAPER SECTION 6.8 - COPY THESE VALUES INTO YOUR PAPER
========================================================================

TABLE 5 VALUES (Mean +/- SD Over 5 Runs):

  Softmax Accuracy:   {sm_acc_m:.2f}% +/- {sm_acc_s:.2f}%
  CatBoost Accuracy:  {cb_acc_m:.2f}% +/- {cb_acc_s:.2f}%

  Softmax AUROC:      {sm_auroc_m:.4f} +/- {sm_auroc_s:.4f}
  CatBoost AUROC:     {cb_auroc_m:.4f} +/- {cb_auroc_s:.4f}

  Softmax F1:         {sm_f1_m:.4f} +/- {sm_f1_s:.4f}
  CatBoost F1:        {cb_f1_m:.4f} +/- {cb_f1_s:.4f}

  Softmax Log Loss:   {sm_ll_m:.4f} +/- {sm_ll_s:.4f}
  CatBoost Log Loss:  {cb_ll_m:.4f} +/- {cb_ll_s:.4f}

TABLE 6 VALUES (Paired t-test):

  Accuracy:  t = {acc_t:.4f}, p = {acc_p:.6f}
  AUROC:     t = {auroc_t:.4f}, p = {auroc_p:.6f}

McNEMAR'S TEST:

  b = {total_b}, c = {total_c}
  chi2 = {agg_chi2:.4f}, p = {agg_p:.6f}

========================================================================
"""
print(text)

with open(f'{output_dir}/Publication_Ready_Text.txt', 'w') as f:
    f.write(text)
!cp {output_dir}/Publication_Ready_Text.txt /content/drive/MyDrive/
print("Saved to Drive!")


## Cell 21: Final Summary


In [ ]:
total_time = sum(catboost_results['time']) + sum(softmax_results['time'])

sm_acc_m = np.mean(softmax_results['accuracy'])*100
sm_acc_s = np.std(softmax_results['accuracy'])*100
cb_acc_m = np.mean(catboost_results['accuracy'])*100
cb_acc_s = np.std(catboost_results['accuracy'])*100
acc_p = float(ttest_results[0]['p-value'])

print("=" * 80)
print("STATISTICAL VALIDATION COMPLETE".center(80))
print("=" * 80)
print(f"""
  Runs completed:        {N_RUNS}
  Seeds:                 {RANDOM_SEEDS}
  Total time:            {total_time/60:.1f} minutes

  KEY RESULTS:
  +-----------------------------------------+
  | Softmax Accuracy:  {sm_acc_m:>6.2f}% +/- {sm_acc_s:.2f}%  |
  | CatBoost Accuracy: {cb_acc_m:>6.2f}% +/- {cb_acc_s:.2f}%  |
  | Improvement:       +{cb_acc_m - sm_acc_m:.2f}%             |
  | Paired t-test:     p = {acc_p:.6f}         |
  +-----------------------------------------+

  Files saved to: statistical_validation_results/
  Files copied to: Google Drive

  Ready for paper submission!
""")
